###

### **Facebook Post Generation Implemented by Iterative workflow**

In [6]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, List, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from operator import add
from pydantic import BaseModel, Field

In [4]:
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model = "gemini-3.6-flash"
)
print(f"[INFO] Model : {model.model} Loaded Successfully!")

[INFO] Model : gemini-3.6-flash Loaded Successfully!


### **Creating PostState**

In [5]:
class PostState(TypedDict):
    topic : str
    post : str
    evaluation: Literal["approved","needs_improvement"]
    feedback : str
    iteration : int
    max_iteration : int

    post_history: Annotated[List[str], add]
    feedback_history: Annotated[List[str], add]
    

### **Creating Post Evaluation Model**


In [8]:
class PostEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final Evaluation Result.")
    feedback: str = Field(..., description="Feedback for the facebook post.")

evaluation_model = model.with_structured_output(PostEvaluation)

### **Creating a PostGenerator Model**

In [14]:
class PostGenerator(BaseModel):
    post : str = Field(..., description="Write a facebook post on the given topic")

post_model = model.with_structured_output(PostGenerator)

### **Creating the Optimizer Model**

In [16]:
class PostOptimizer(BaseModel):
    post : str = Field(..., description="Optimize the given Facebook Post")

optimizer_model = model.with_structured_output(PostOptimizer)

### **Node 1 : Generation Node**

In [18]:
def post_generation(state: PostState):
    topic = state['topic']

    prompt = [
        SystemMessage(content="You are a funny and clever facebook post influencer."),
        HumanMessage(content= f"""Write a short, original, and hilarious Facebook post on the topic: "{topic}".\n
        Rules:
        - Do NOT use question-answer format.
        - Max 200 words.
        - Use observational humor, irony, sarcasm, or cultural references.
        - Think in meme logic, punchlines, or relatable takes.
        - Use simple, day to day english
        """
        ) 
    ]

    post = post_model.invoke(prompt).post

    return {
        "post" : post,
        "post_history" : [post]
    }

### **Node 2 : Evaluate Post Node**